# 02 · 动态调整与状态恢复

第一轮从资料获得补充约束；第二轮用户要求增加验收清单。
用 `InMemorySaver` 观察同会话继续、重建 Agent 后读取状态、新 thread 隔离。

这是已结束一轮调用后的会话继续，不演示执行中断、进程重启或事务恢复。

In [ ]:
from planning_lab import (
    initialize, live_model, TOOLS, SYSTEM_PROMPT, REQUEST, FOLLOWUP,
    read_requirements, read_constraints, observe_update, final_answer,
)
initialize()
# 保持 False 可从头执行资料与 schema 检查，不调用模型。
# 确认密钥后手动改成 True，再从头运行。本章不会使用模拟模型。
RUN_LIVE = False
print(read_requirements.invoke({}))
print(read_constraints.invoke({}))
for tool in TOOLS:
    print(tool.name, tool.args_schema.model_json_schema())

In [ ]:
from uuid import uuid4
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langgraph.checkpoint.memory import InMemorySaver

saver = InMemorySaver()
config = {'configurable': {'thread_id': 'ch10-' + uuid4().hex}, 'recursion_limit': 60}

## 第一轮：读取资料并调整工作待办（真实调用）

In [ ]:
if RUN_LIVE:
    model = live_model()
    agent = create_agent(
        model=model, tools=TOOLS, system_prompt=SYSTEM_PROMPT,
        middleware=[TodoListMiddleware()], checkpointer=saver,
    )
    trace = {'messages': [], 'snapshots': [], 'todos': []}
    for update in agent.stream(
        {'messages': [{'role': 'user', 'content': REQUEST}]},
        config, stream_mode='updates',
    ):
        observe_update(update, trace)
    first_state = agent.get_state(config)
    print('保存的待办：', first_state.values.get('todos'))
else:
    print('已跳过真实调用。')

## 第二轮：重建 Agent 并继续同一 thread（真实调用）

必须复用**同一个 saver 实例**和 `thread_id`。重新创建 `InMemorySaver()` 就没有这些检查点。

第二轮只传新增用户消息，不把完整旧消息重新追加一次。

In [ ]:
if RUN_LIVE:
    rebuilt = create_agent(
        model=model, tools=TOOLS, system_prompt=SYSTEM_PROMPT,
        middleware=[TodoListMiddleware()], checkpointer=saver,
    )
    restored = rebuilt.get_state(config)
    assert restored.values == first_state.values
    assert restored.next == ()  # 前一轮已经结束，不是挂起待恢复的工具调用。
    print('重建后已读取相同状态；接下来才提交用户新要求。')
    follow_trace = {'messages': [], 'snapshots': [], 'todos': []}
    for update in rebuilt.stream(
        {'messages': [{'role': 'user', 'content': FOLLOWUP}]},
        config, stream_mode='updates',
    ):
        observe_update(update, follow_trace)
    second_state = rebuilt.get_state(config)
    assert len(second_state.values['messages']) > len(first_state.values['messages'])
    print('更新后的待办：', second_state.values.get('todos'))
    print('包含验收清单的新方案：', final_answer(follow_trace))

## 新 thread 没有旧会话状态（不额外调用模型）

In [ ]:
if RUN_LIVE:
    fresh_config = {'configurable': {'thread_id': 'fresh-' + uuid4().hex}}
    fresh = rebuilt.get_state(fresh_config)
    assert not fresh.values.get('messages')
    assert not fresh.values.get('todos')
    print('新 thread 没有继承旧消息与待办。')

## 观察重点

- 第一轮补充资料是否引起待办内容变化，而不仅是状态变化？
- 第二轮是否产生跟验收有关的新工作，并在最终回答交付清单？
- 检查旧的手机号验证码、密码暂缓和 SQLite 约束是否被保留。
- `get_state()` 得到检查点中的完整状态；`updates` 得到节点增量；模型请求只包含框架和中间件实际组装的内容。这三者不能混用。
- 待办列表允许整表替换。是否保留所有旧已完成条目是提示词与应用设计问题，默认工具没有历史审计保证。
- 进程关闭后 `InMemorySaver` 的数据就消失；同 saver 同 thread 也不提供外部工具事务或安全重放保证。